## Basic Generate

In [1]:
import httpx

with httpx.Client(
    proxy=None,
    trust_env=False
) as client:
    response = client.post(
        "http://127.0.0.1:8000/basic_generate",
        json={
            "prompt": "What is the capital of the United States?"
        },
        timeout=60.0
    )

print(response.status_code)
print(response.text)

200
{"generated_text":"What is the capital of the United States? The capital of the United States is Washington, D.C. (formerly known as the District of Columbia). It is located on the National Mall in the heart of the nation's capital city.\n\nWashington, D.C., officially called the District of Columbia,"}


## Batch Generate

In [2]:
import httpx

payload = {
    "prompts": [
        "The capital of France is",
        "The capital of India is",
        "The capital of Japan is",
        "The largest planet is"
    ]
}

with httpx.Client(trust_env=False) as client:
    response = client.post(
        "http://127.0.0.1:8000/generate",
        json=payload,
        timeout=60.0
    )

print(response.status_code)
print(response.json())

200
{'generated_texts': ['The capital of France is Paris. Which of the following statements about Paris is incorrect?\nA. It was originally named "Paris" after the Roman emperor.\nB. The city has a long history, dating back to the Neolithic Age.\nC. It is located in the', "The capital of India is located in the state of ____.\nA. Madhya Pradesh\nB. Uttar Pradesh\nC. Rajasthan\nD. Maharashtra\n\nTo determine which state's capital is located in the Indian state of Uttar Pradesh, let's follow these steps:\n\n1.", 'The capital of Japan is located on the Pacific coast, in a region known as the ___.\nA. North\nB. South\nC. East\nD. West\n\nTo determine the correct answer to the question "The capital of Japan is located on the Pacific coast,', 'The largest planet isFind the largest planet in our solar system based on its mass. The largest planet is Mercury.\n\n### Step-by-Step Reasoning:\n\n1. **Understanding Mass**: \n   - The mass of a planet is an important factor in determining its size a

## Concurrent Request from multiple clients

In [3]:
import asyncio
import httpx
import time

prompts = [
    "What is KV cache?",
    "What is attention?",
    "What is speculative decoding?",
    "What is continuous batching?"
]

async def call_api(prompt):
    async with httpx.AsyncClient(trust_env=False) as client:
        start = time.perf_counter()

        r = await client.post(
            "http://127.0.0.1:8000/basic_generate",
            json={"prompt": prompt},
            timeout=60.0
        )

        elapsed = time.perf_counter() - start

        return {
            "prompt": prompt,
            "time": round(elapsed, 2),
            "response": r.json()
        }

results = await asyncio.gather(
    *[call_api(p) for p in prompts]
)

for r in results:
    print(r)

{'prompt': 'What is KV cache?', 'time': 1.61, 'response': {'generated_text': 'What is KV cache? What are the benefits of using it?\n\nThe KVM (Kernel-based Virtual Machine) cache is a special data structure used by KVM to store and retrieve information about virtual machines, such as their IP addresses, MAC addresses, and other configuration details.'}}
{'prompt': 'What is attention?', 'time': 3.04, 'response': {'generated_text': "What is attention? Attention is the mental state or process of focusing one's attention on a particular object, idea, or activity. It is a fundamental aspect of human cognition and can be divided into three types: internal (involuntary), external (involuntary),"}}
{'prompt': 'What is speculative decoding?', 'time': 4.45, 'response': {'generated_text': 'What is speculative decoding? What are the main applications of it?\n\nSpeculative Decoding, also known as speculative decoding or speculative analysis, refers to a method used in cryptography and computer scie

## Latency - KV Cache (same question - multiple iteration)
1. Post first time, the response is faster

In [4]:
import time
import httpx

with httpx.Client(trust_env=False) as client:
    start = time.perf_counter()

    response = client.post(
        "http://127.0.0.1:8000/basic_generate",
        json={
            "prompt": "Explain KV cache in 3 sentences."
        },
        timeout=60.0
    )

    end = time.perf_counter()

print("Latency:", round(end - start, 3), "seconds")
print(response.json())

Latency: 1.478 seconds
{'generated_text': "Explain KV cache in 3 sentences. A key-value (KV) cache is a type of memory that stores data in an efficient way to quickly retrieve information from, but it's not as fast as a database or file system.\nIt uses a hash table to store key-value pairs and provides"}
